In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/MINI_PROJECT/DAiSEE/DataSet"

Test  Test.txt	Train  Train.txt  Validation  Validation.txt


In [ ]:
%cd /content/drive/MyDrive/MINI_PROJECT/DAiSEE

/content/drive/.shortcut-targets-by-id/1juEhlfoTVwLCaEKw1JRlOUwEKEUhl0Or/MINI_PROJECT/DAiSEE


In [ ]:
!ls

DataSet  extractFrames.py  GenderClips	hog.py	Labels	README.txt


In [ ]:
!python extractFrames.py

Mini Project Frame Extraction Done


In [ ]:
!find DataSet/Train -name "*.jpg" | wc -l

^C


In [ ]:
!ls DataSet/Train | head

110001
110002
110004
110005
110006
110007
110008
110010
110011
110012


In [ ]:
!find DataSet/Train -name "*.jpg" | head

DataSet/Train/110013/1100131011/1100131011_0001.jpg
DataSet/Train/110013/1100131011/1100131011_0002.jpg
DataSet/Train/110013/1100131011/1100131011_0003.jpg
DataSet/Train/110013/1100131011/1100131011_0004.jpg
DataSet/Train/110013/1100131011/1100131011_0005.jpg
DataSet/Train/110013/1100131011/1100131011_0006.jpg
DataSet/Train/110013/1100131011/1100131011_0007.jpg
DataSet/Train/110013/1100131011/1100131011_0008.jpg
DataSet/Train/110013/1100131011/1100131011_0009.jpg
DataSet/Train/110013/1100131011/1100131011_0010.jpg


In [ ]:
!head DataSet/Train.txt

1100011002.avi
1100011003.avi
1100011004.avi
1100011005.avi
1100011006.avi
1100011007.avi
1100011008.avi
1100011009.avi
1100011010.avi
1100011011.avi


In [ ]:
!ls Labels

AllLabels.csv  TestLabels.csv  TrainLabels.csv	ValidationLabels.csv


In [ ]:
import pandas as pd

df = pd.read_csv("Labels/TrainLabels.csv")
df.head()

,ClipID,Boredom,Engagement,Confusion,Frustration
0,1100011002.avi,0,2,0,0
1,1100011003.avi,0,2,0,0
2,1100011004.avi,0,3,0,0
3,1100011005.avi,0,3,0,0
4,1100011006.avi,0,3,0,0


In [ ]:
import os
import pandas as pd

# Load labels
df = pd.read_csv("Labels/TrainLabels.csv")

# Keep only ClipID and Engagement
df = df[["ClipID", "Engagement"]]

# Remove .avi from ClipID
df["ClipID"] = df["ClipID"].str.replace(".avi", "", regex=False)

image_paths = []
labels = []

train_path = "DataSet/Train"

for user in os.listdir(train_path):
    user_path = os.path.join(train_path, user)

    for clip in os.listdir(user_path):
        clip_path = os.path.join(user_path, clip)

        # Get engagement label
        label_row = df[df["ClipID"] == clip]

        if len(label_row) == 0:
            continue

        engagement_label = int(label_row["Engagement"].values[0])

        for img in os.listdir(clip_path):
            if img.endswith(".jpg"):
                image_paths.append(os.path.join(clip_path, img))
                labels.append(engagement_label)

print("Total images:", len(image_paths))

Total images: 7150


In [ ]:
import shutil
import os

source_root = "DataSet/Train"
dest_root = "project/data/extracted_frames"

os.makedirs(dest_root, exist_ok=True)

for root, dirs, files in os.walk(source_root):
    for file in files:
        if file.endswith(".jpg"):
            src_path = os.path.join(root, file)
            dst_path = os.path.join(dest_root, file)
            shutil.copy(src_path, dst_path)

print("Frames copied successfully.")

Frames copied successfully.


In [ ]:
import cv2
import os

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

input_folder = "project/data/extracted_frames"
output_folder = "project/data/cropped_faces"

os.makedirs(output_folder, exist_ok=True)

for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)
    img = cv2.imread(img_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = img[y:y+h, x:x+w]
        cv2.imwrite(os.path.join(output_folder, img_name), face)
        break  # take first detected face only

print("Face cropping done.")

Face cropping done.


In [ ]:
input_folder = "project/data/cropped_faces"
output_folder = "project/data/processed_faces"

os.makedirs(output_folder, exist_ok=True)

for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)
    img = cv2.imread(img_path)

    if img is None:
        continue

    img = cv2.resize(img, (224, 224))
    cv2.imwrite(os.path.join(output_folder, img_name), img)

print("Preprocessing done.")

Preprocessing done.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.DataFrame({
    "image_path": [f"project/data/processed_faces/{img}"
                   for img in os.listdir("project/data/processed_faces")],
    "label": labels[:len(os.listdir("project/data/processed_faces"))]
})

train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df.to_csv("project/data/train.csv", index=False)
val_df.to_csv("project/data/val.csv", index=False)
test_df.to_csv("project/data/test.csv", index=False)

print("CSV splits created.")

CSV splits created.


In [ ]:
import os

count = 0
for root, dirs, files in os.walk("data/processed_faces"):
    for file in files:
        if file.endswith(".jpg"):
            count += 1

print("Total processed images:", count)

Total processed images: 0


In [ ]:
import os

print(os.listdir())

['extractFrames.py', 'hog.py', 'README.txt', 'GenderClips', 'Labels', 'DataSet', 'project']


In [ ]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".jpg"):
            print(os.path.join(root, file))

Streaming output truncated to the last 5000 lines.
./DataSet/Train/210057/2100572027/2100572027_0001.jpg
./DataSet/Train/210057/2100572027/2100572027_0002.jpg
./DataSet/Train/210057/2100572027/2100572027_0003.jpg
./DataSet/Train/210057/2100572027/2100572027_0004.jpg
./DataSet/Train/210057/2100572027/2100572027_0005.jpg
./DataSet/Train/210057/2100572027/2100572027_0006.jpg
./DataSet/Train/210057/2100572027/2100572027_0007.jpg
./DataSet/Train/210057/2100572027/2100572027_0008.jpg
./DataSet/Train/210057/2100572027/2100572027_0009.jpg
./DataSet/Train/210057/2100572027/2100572027_0010.jpg
./DataSet/Train/210057/2100572041/2100572041_0001.jpg
./DataSet/Train/210057/2100572041/2100572041_0002.jpg
./DataSet/Train/210057/2100572041/2100572041_0003.jpg
./DataSet/Train/210057/2100572041/2100572041_0004.jpg
./DataSet/Train/210057/2100572041/2100572041_0005.jpg
./DataSet/Train/210057/2100572041/2100572041_0006.jpg
./DataSet/Train/210057/2100572041/2100572041_0007.jpg
./DataSet/Train/210057/21005720

KeyboardInterrupt: 

In [ ]:
print(os.getcwd())

/content/drive/.shortcut-targets-by-id/1juEhlfoTVwLCaEKw1JRlOUwEKEUhl0Or/MINI_PROJECT/DAiSEE


In [ ]:
os.chdir("project")

In [ ]:
len(os.listdir("data/processed_faces"))

6748

In [ ]:
import os
print(os.getcwd())

/content/drive/.shortcut-targets-by-id/1juEhlfoTVwLCaEKw1JRlOUwEKEUhl0Or/MINI_PROJECT/DAiSEE/project


In [ ]:
print("Extracted:", len(os.listdir("data/extracted_frames")))
print("Cropped:", len(os.listdir("data/cropped_faces")))
print("Processed:", len(os.listdir("data/processed_faces")))

Extracted: 7950
Cropped: 6748
Processed: 6748


In [ ]:
import pandas as pd

labels_path = "../Labels/TrainLabels.csv"   # adjust if name is different
df = pd.read_csv(labels_path)

df.head()

,ClipID,Boredom,Engagement,Confusion,Frustration
0,1100011002.avi,0,2,0,0
1,1100011003.avi,0,2,0,0
2,1100011004.avi,0,3,0,0
3,1100011005.avi,0,3,0,0
4,1100011006.avi,0,3,0,0


In [ ]:
import os

image_folder = "data/processed_faces"
images = os.listdir(image_folder)

data = []

for img in images:
    clip_id = img.split("_")[0] + ".avi"   # reconstruct original ClipID

    row = df[df["ClipID"] == clip_id]

    if not row.empty:
        engagement_label = row["Engagement"].values[0]   # example target
        data.append([os.path.join(image_folder, img), engagement_label])

final_df = pd.DataFrame(data, columns=["image_path", "label"])

print("Total matched images:", len(final_df))

Total matched images: 5967


In [ ]:
print("Total processed images:", len(images))

Total processed images: 6748


In [ ]:
print("Total label clips loaded:", len(df))
print("Sample ClipIDs:", df["ClipID"].head())

Total label clips loaded: 5358
Sample ClipIDs: 0    1100011002.avi
1    1100011003.avi
2    1100011004.avi
3    1100011005.avi
4    1100011006.avi
Name: ClipID, dtype: object


In [ ]:
final_df["label"].value_counts()

,count
label,
2,3228
3,2502
1,234
0,3


In [ ]:
final_df = final_df[final_df["label"] != 0]

print(final_df["label"].value_counts())

label
2    3228
3    2502
1     234
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    final_df,
    test_size=0.3,
    random_state=42,
    stratify=final_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 4174
Validation: 895
Test: 895


In [ ]:
train_df.to_csv("data/train.csv", index=False)
val_df.to_csv("data/val.csv", index=False)
test_df.to_csv("data/test.csv", index=False)

print("CSV files saved successfully.")

CSV files saved successfully.


In [ ]:
import os
print(os.listdir("data"))

['extracted_frames', 'cropped_faces', 'processed_faces', 'train.csv', 'val.csv', 'test.csv']


In [ ]:
train_df["label"].value_counts()

,count
label,
2,2259
3,1751
1,164


In [ ]:
import os
os.makedirs("src", exist_ok=True)
print("src folder created")

src folder created


In [ ]:
open("src/extract_frames.py", "w").close()
open("src/face_detection.py", "w").close()
open("src/preprocessing.py", "w").close()
open("src/dataset_split.py", "w").close()

print("Python files created")

Python files created
